# 🗡️ YOLOv26-Based Kris Detection and Classification for Madura Cultural Heritage Preservation
## Notebook Uji Coba Model + Google Colab Live Webcam Inference

Notebook ini dikembangkan khusus untuk melatih dan menguji model **YOLOv26** menggunakan GPU T4 Google Colab secara langsung dari repositori hasil anotasi.

---

## 1. 📂 Clone Repositori Penelitian & Setup

In [ ]:
# 1. Kloning repositori kode dan knowledge base
# Ganti URL repositori dengan tautan repo GitHub Anda
!git clone https://github.com/username/yolov26-madura-kris-preservation.git
%cd yolov26-madura-kris-preservation

In [ ]:
# 2. Verifikasi akselerator GPU T4 aktif
!nvidia-smi

# 3. Pasang library ultralytics dan bs4
!pip install -q 'ultralytics>=8.3.50' beautifulsoup4 pillow requests pandas tqdm opencv-python matplotlib

## 2. 📦 Import Annotated Dataset

Untuk melatih model YOLOv26 di Google Colab, Anda perlu mengunggah folder dataset `dataset_keris` yang telah dianotasi (berisi folder `/images` dan `/labels` yang dihasilkan dari Dashboard Web).

In [ ]:
# Opsi A: Hubungkan ke Google Drive untuk mengambil file dataset_keris.zip
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/dataset_keris.zip -d ./dataset_keris

# Opsi B: Buat dummy folder latihan jika belum ada dataset real (agar script tidak error)
import os
os.makedirs("./dataset_keris/images/train", exist_ok=True)
os.makedirs("./dataset_keris/labels/train", exist_ok=True)
print("Direktori dataset siap!")

## 3. 🏋️ YOLOv26 Training Pipeline

In [ ]:
# Buat file konfigurasi dataset YAML untuk Ultralytics
dataset_yaml = """
path: ../dataset_keris
train: images/train
val: images/train # fallback ke train folder jika data sedikit

names:
  0: keris_lurus
  1: keris_luk_3
  2: keris_luk_5
  3: keris_luk_7
  4: keris_luk_9
  5: keris_luk_11
  6: keris_luk_13
  7: keris_madura
  8: keris_majapahit
  9: keris_mataram
  10: keris_unknown
"""

with open("dataset_keris.yaml", "w") as f:
    f.write(dataset_yaml.strip())
print("dataset_keris.yaml siap di-load oleh YOLOv26.")

In [ ]:
from ultralytics import YOLO

# Muat arsitektur model YOLOv26
model = YOLO('yolo26n.pt')

# Jalankan training pada GPU
# Uncomment baris di bawah setelah Anda mengunggah dataset beranotasi lengkap
# model.train(data='dataset_keris.yaml', epochs=30, imgsz=640, device=0)

## 4. 📷 Google Colab Live Webcam Capture & Inference

Blok kode di bawah menggunakan JavaScript untuk mengakses webcam browser laptop Anda di Google Colab, mengambil gambar, dan memprosesnya secara real-time dengan YOLOv26.

In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
from PIL import Image
import io

def take_photo(filename='photo.jpg', quality=0.8):
  # JavaScript untuk membuat live video feed dan tombol capture di dalam cell output Colab
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture Frame';
      capture.style.background = '#C9A84C';
      capture.style.color = '#000';
      capture.style.border = 'none';
      capture.style.padding = '8px 16px';
      capture.style.borderRadius = '4px';
      capture.style.fontWeight = 'bold';
      capture.style.cursor = 'pointer';
      capture.style.margin = '10px 0';
      
      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.borderRadius = '8px';
      const stream = await navigator.mediaDevices.getUserMedia({video: {facingMode: 'environment'}});
      
      document.body.appendChild(div);
      div.appendChild(video);
      div.appendChild(capture);
      video.srcObject = stream;
      await video.play();
      
      // Resize the output to fit inside the cell nicely
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      
      // Tunggu tombol diclick untuk mengambil gambar
      await new Promise((resolve) => capture.onclick = resolve);
      
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

## 5. 🏛️ Jalankan Deteksi Kamera & Tampilkan Metadata Budaya

In [ ]:
# Load basis pengetahuan budaya lokal
kb_path = "backend/knowledge_base.json"
cultural_db = {}
if os.path.exists(kb_path):
    with open(kb_path) as f:
        cultural_db = json.load(f)

def show_cultural_heritage_card(label, conf):
    """Render HTML metadata card untuk estetika visual di Colab."""
    kb = cultural_db
    if not kb:
        print(f"Detected: {label} (Conf: {conf:.2%})")
        return
        
    # Get Dapur info
    d_key = "tilam_upih" if "lurus" in label else "sengkelat" if "13" in label else "carita"
    d = kb.get("dapur", {}).get(d_key, {"nama": "Jalak", "deskripsi": "Bilah khas Madura", "filosofi": "Kesederhanaan"})
    p = kb.get("pamor", {}).get("beras_wutah", {"nama": "Beras Wutah", "makna": "Kemakmuran", "proses": "Tempa mlumah"})
    e = kb.get("empu", {}).get("empu_aeng_tongtong", {"nama": "Empu Aeng Tongtong", "era": "Abad ke-17", "asal": "Sumenep"})
    t = kb.get("tangguh", {}).get("madura", {"nama": "Tangguh Madura", "periode": "Modern"})
    sb = kb.get("status_budaya", {"unesco": "UNESCO Heritage 2008"})

    card_html = f"""
    <div style='background: linear-gradient(135deg, #1a0e00 0%, #2d1a00 100%); border: 2px solid #C9A84C; border-radius: 8px; padding: 16px; color: #f5e6c8; max-width: 550px; font-family: sans-serif; margin: 10px 0;'>
      <h3 style='margin: 0 0 10px; color: #C9A84C; border-bottom: 1px solid #C9A84C33; padding-bottom: 6px;'>🗡️ IDENTIFIKASI BUDAYA MADURA</h3>
      <div style='font-size: 11px; color: #C9A84C; text-transform: uppercase; margin-bottom: 2px;'>Akurasi YOLOv26</div>
      <div style='font-size: 16px; font-weight: bold; margin-bottom: 12px;'>{label} ({conf:.1%})</div>
      <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 10px; font-size: 12px; margin-bottom: 12px;'>
        <div><b>⚔️ Dapur:</b> {d['nama']}</div>
        <div><b>✨ Pamor:</b> {p['nama']}</div>
        <div><b>🏛️ Empu:</b> {e['nama']}</div>
        <div><b>📅 Tangguh:</b> {t['nama']}</div>
      </div>
      <div style='font-size: 11px; background: rgba(201,168,76,0.08); padding: 8px; border-left: 3px solid #C9A84C; border-radius: 0 4px 4px 0; margin-bottom: 10px;'>
        <b>Makna Filosofis:</b> {d.get('filosofi', 'Simbol kejujuran dan kekuatan batin.')}
      </div>
      <div style='font-size: 10px; color: #90caf9;'>🌐 <b>UNESCO:</b> {sb.get('unesco', '')}</div>
    </div>
    """
    display(HTML(card_html))

try:
  # 1. Buka webcam untuk capture satu frame
  photo_file = take_photo()
  
  # 2. Run inference menggunakan model
  # (Menggunakan model pretrained jika model fine-tuned belum selesai dilatih)
  model_detect = YOLO('yolo26n.pt')
  img = cv2.imread(photo_file)
  results = model_detect.predict(img, conf=0.25, verbose=False)
  
  # Draw bounding boxes
  for r in results:
      boxes = r.boxes
      for box in boxes:
          x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
          conf = float(box.conf[0].item())
          cls = int(box.cls[0].item())
          name = r.names.get(cls, "keris_unknown")
          
          # Draw on image
          cv2.rectangle(img, (x1, y1), (x2, y2), (76, 168, 201), 3)
          cv2.putText(img, f"{name} {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (76, 168, 201), 2)
          
          # Render cultural cards
          show_cultural_heritage_card(name, conf)
          
  # Tampilkan hasil gambar yang terdeteksi
  plt.figure(figsize=(10, 8))
  plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()
  
except Exception as err:
  print(str(err))
